# Ayanami Phase 10.5 training (A/B) — Colab
Run A: text_sft (teacher text only). Run B: logit_kd (cached top-32).
One run per session is fine (~1h each on T4); checkpoints resume.
Run A first, then B. Adapters + metrics go to Drive.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
REPO = '/content/Ayanami'
if not os.path.isdir(REPO):
    !git clone https://github.com/Sarahi-M-00/Ayanami.git /content/Ayanami
%cd /content/Ayanami
!git pull --ff-only 2>&1 | tail -1


In [ ]:
# One-time per VM (~10 min). Re-run after every VM recycle.
!sudo apt-get update -qq && sudo apt-get install -qq -y python3.13-venv
!rm -rf .venv
!bash scripts/setup_remote.sh  # ends with: check_env PASSED (student_base FAILs expected here)

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN set from Colab Secrets')
except Exception as exc:
    print('set HF_TOKEN first:', exc)
%env HF_HOME=/tmp/hf_cache
from google.colab import drive; drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/ayanami/runs/'
!mkdir -p "$OUT"
print('OUT =', OUT)


In [ ]:
# Student weights (Apache-2.0, pinned revision from configs/student.yaml).
# Skips if already downloaded in a previous session.
import os
os.makedirs('student_base', exist_ok=True)
if not os.path.exists('student_base/model.safetensors'):
    !.venv/bin/python -c "from huggingface_hub import snapshot_download; snapshot_download('orlandorubino/Qwen3-1.7B-heretic', revision='433869992df66cd5ed9fa77491809448c3dac7a0', local_dir='student_base')"
else:
    print('student_base already present')
!ls -la student_base/ | head


In [ ]:
# Upload train_v1.jsonl (35MB) + dev_v1.jsonl (3MB) from the laptop, once per VM.
# Files land in data/processed/. Skip if already there.
import os
os.makedirs('data/processed', exist_ok=True)
need = [f for f in ('data/processed/train_v1.jsonl', 'data/processed/dev_v1.jsonl')
        if not os.path.exists(f)]
if need:
    print('upload these files:', need)
    from google.colab import files
    up = files.upload()
    for name, data in up.items():
        dest = 'data/processed/' + name
        open(dest, 'wb').write(data)
        print('saved', dest, len(data))
else:
    print('train/dev already present')
!ls -la data/processed/train_v1.jsonl data/processed/dev_v1.jsonl


In [ ]:
# STOP GATE: student tokenizer fingerprint must equal the teacher's.
# Run B is INVALID if this fails. Do not skip.
import sys; sys.path.insert(0, 'scripts')
from transformers import AutoTokenizer
from teacher_cache import fingerprint_of_tokenizer
fp = fingerprint_of_tokenizer(AutoTokenizer.from_pretrained('student_base', local_files_only=True))
print('student fp:', fp)
assert fp == '563a701b87f87d8076a6faf47ea67cc1292b2321e25f18d09e2d11b6189140aa', 'FINGERPRINT MISMATCH — STOP'
print('fingerprint OK — Run B allowed')


In [ ]:
# Run A: text_sft (~1h on T4). Resumes from checkpoints if re-run.
!PYTHONPATH=src .venv/bin/python -m ayanami_distill.train --config configs/distill/phase10_text_sft.yaml --run-dir runs/phase10_text_sft

In [ ]:
# Run B: logit_kd (~1-1.5h on T4). Run in this or a later session.
!PYTHONPATH=src .venv/bin/python -m ayanami_distill.train --config configs/distill/phase10_logit_kd.yaml --run-dir runs/phase10_logit_kd

In [ ]:
!cp -r runs/phase10_text_sft "$OUT" 2>/dev/null; !cp -r runs/phase10_logit_kd "$OUT" 2>/dev/null; !ls "$OUT"
# Copy the new adapters + metrics.jsonl + summary.json back to the laptop.
# Record GPU hours in docs/GPU_LEDGER.md and tell Luz.
